# Comprehensive Technical Report: Multi-Timescale Compressed Learning Framework for ECG Diagnosis

---

## 1. Introduction and Architectural Motivation

Traditional automated Electrocardiogram (ECG) telemetry architectures deployed on battery-powered Internet-of-Medical-Things (IoMT) devices suffer from two fundamental bottlenecks:

1. **The Telemetry Energy Bottleneck:** Continuous raw transmission of multi-lead Nyquist-rate ECG signals drains wearable batteries rapidly. While Compressive Sensing (CS) drastically reduces transmission payload via linear projections ($\mathbf{y} = \mathbf{\Phi}\mathbf{x}$), conventional architectures rely on server-side iterative decompression (such as Orthogonal Matching Pursuit [OMP] or Basis Pursuit De-Noising [BPDN]). These reconstruction algorithms introduce severe reconstruction latency and cannot easily scale for real-time edge alerting.
2. **The Single-Beat Morphology Ceiling:** Direct classification in the compressed domain (Compressed Learning) avoids reconstruction. However, existing single-beat morphology paradigms exhibit an acute diagnostic blind spot: **Congestive Heart Failure (CHF) heartbeats frequently present identical local ventricular morphology to Normal Sinus Rhythm (NSR) beats**. Relying solely on isolated heartbeat morphology results in severe diagnostic leakage (>20% false-negative missed CHF cases), misclassifying dangerous failure patterns as benign healthy sinus rhythm.

To resolve these challenges simultaneously, this framework introduces a **Reconstruction-Free, Multi-Timescale Compressed Learning Architecture**. Instead of heavy 1D-CNNs or Transformer networks that demand hundreds of thousands of parameters, the classification backbone is formulated as an ultra-lightweight Multi-Layer Perceptron (MLP) multi-branch system that operates on three synchronized physiological timescales.

---

## 2. Dataset Ingestion and Preprocessing Pipeline

### 2.1 Benchmark Data Sources
The framework benchmarks across three foundational PhysioNet cardiology databases:

* **MIT-BIH Arrhythmia Database (`mitdb`):** 48 half-hour two-channel ambulatory ECG recordings sampled natively at $360\text{ Hz}$.
* **MIT-BIH Normal Sinus Rhythm Database (`nsrdb`):** 18 long-term recordings of subjects with no significant cardiac arrhythmias, sampled natively at $128\text{ Hz}$.
* **BIDMC Congestive Heart Failure Database (`chfdb`):** 15 long-term recordings from severe heart failure subjects (NYHA Class III–IV), sampled natively at $250\text{ Hz}$.

### 2.2 Standardized Multi-Stage Preprocessing
To establish rigorous consistency across disparate hardware sources:

1. **Temporal Resampling:** All signals are resampled using polyphase anti-aliasing filter interpolation to a uniform sampling frequency:
   $$f_s = 128\text{ Hz}$$
   This yields a temporal sampling interval of $T_s = 7.8125\text{ ms}$.

2. **Denoising and Baseline Wander Removal:**
   * High-frequency electromyographic (EMG) and powerline artifacts ($50/60\text{ Hz}$) are attenuated via a zero-phase 4th-order Butterworth bandpass filter ($0.5\text{ Hz} - 45.0\text{ Hz}$).
   * Baseline wander induced by respiration and subject movement is suppressed using dual sequential median filtering: a $200\text{ ms}$ window extracts the QRS complex and P/T waves, followed by a $600\text{ ms}$ window to compute the baseline wander profile, which is subtracted from the filtered signal.
   * Fine smoothing is completed with a Savitzky-Golay polynomial filter (window length $11$, order $3$).

3. **Segmentation & Amplitude Standardization:**
   * Heartbeats are segmented into uniform windows of length $N = 128$ samples centered around the detected R-peak ($48$ samples pre-R, $79$ samples post-R, corresponding to a $1.0\text{-second}$ temporal capture window at $128\text{ Hz}$).
   * Each extracted beat vector $\mathbf{x} \in \mathbb{R}^{128}$ is normalized via Z-score standardization:
     $$\mathbf{x}_{\text{norm}} = \frac{\mathbf{x} - \mu_{\mathbf{x}}}{\sigma_{\mathbf{x}} + \epsilon}$$

---

## 3. Multi-Scale Feature Engineering Pipeline

The system constructs three synchronized feature domains corresponding to micro-, meso-, and macro-timescales:
### 3.1 Micro-Scale: Sub-Nyquist Compressed Morphology ($\mathbf{y} \in \mathbb{R}^M$)

The normalized beat vector $\mathbf{x} \in \mathbb{R}^{128}$ is directly projected onto a low-dimensional measurement subspace without signal reconstruction:

$$\mathbf{y} = \mathbf{\Phi}\mathbf{x}$$

$\mathbf{\Phi} \in \mathbb{R}^{M \times 128}$ is a fixed, Gaussian random projection measurement matrix:

$$\Phi_{i,j} \sim \mathcal{N}\left(0, \frac{1}{M}\right)$$

Sub-Nyquist evaluation sweeps through $M \in \{32, 64, 96, 128\}$, where $M=32$ achieves an aggressive Compression Ratio (CR) of $75\%$:

$$CR = 1 - \frac{M}{N}$$

In addition to data compression, $\mathbf{\Phi}$ acts as an implicit spatial regularizer that filters high-frequency sensor noise.

---

### 3.2 Meso-Scale: Local Coupling Rhythms ($\mathbf{r} \in \mathbb{R}^4$)

Local beat timing dynamics are captured using instantaneous R-R interval relationships:

$$\mathbf{r} = \left[ \text{RR}_{\text{pre}},\, \text{RR}_{\text{post}},\, \frac{\text{RR}_{\text{pre}}}{\text{RR}_{\text{post}}},\, \frac{\text{RR}_{\text{local\_avg}}}{\text{RR}_{\text{pre}}} \right]^T$$

- $\text{RR}_{\text{pre}} = t_i - t_{i-1}$: Pre-coupling interval  
- $\text{RR}_{\text{post}} = t_{i+1} - t_i$: Post-coupling interval  

Dynamic ratio captures premature ventricular contractions (PVCs) and compensatory pauses.  
Local moving average (10-beat window) contextualizes ectopic deviations against background heart rate.

---

### 3.3 Macro-Scale: Autonomic Heart Rate Variability ($\mathbf{h} \in \mathbb{R}^8$)

To resolve the morphology ceiling where CHF heartbeats mimic healthy NSR, continuous autonomic nervous system tone is captured over a rolling $60\text{-second}$ window preceding each heartbeat:

$$\mathbf{h} = [\text{MeanRR},\, \text{SDNN},\, \text{RMSSD},\, \text{pNN50},\, \text{VLF},\, \text{LF},\, \text{HF},\, \text{LF/HF}]^T$$

**Time Domain Features:**
- **SDNN:** Overall autonomic variability  
- **RMSSD:** Root mean square of successive RR differences; direct indicator of parasympathetic/vagal tone (significantly depressed in severe CHF)  
- **pNN50:** Percentage of adjacent RR intervals differing by $>50\text{ ms}$  

**Frequency Domain Features (via Lomb-Scargle Periodogram on unevenly spaced intervals):**
- **VLF (0.0033–0.04 Hz):** Long-term regulatory mechanisms  
- **LF (0.04–0.15 Hz):** Sympathetic and parasympathetic mixed modulation  
- **HF (0.15–0.40 Hz):** Parasympathetic efferent activity  
- **LF/HF Ratio:** Quantifies sympathovagal imbalance  

---

## 4. Rigorous Inter-Patient Benchmark Protocol

To prevent misleading and over-optimistic performance estimates caused by random beat shuffling (intra-patient data leakage), data partitioning strictly follows the **AAMI/de Chazal inter-patient protocol**:

- No patient record appears in both the training and test cohorts simultaneously.  
- The test set consists of **101,736 unseen cardiac beats** across three diagnostic classes:  
  - **Class 0 (NSR):** Normal Sinus Rhythm  
  - **Class 1 (ARR):** Cardiac Arrhythmias (Ectopic beats, PVC, PAC, etc.)  
  - **Class 2 (CHF):** Congestive Heart Failure  

---

## 5. Neural Network Architectures: Dual-Branch vs. Tri-Branch

Unlike heavy convolutional models, the classifier backbone is constructed entirely using lightweight, specialized feedforward **Multi-Layer Perceptrons (MLPs)**.
### 5.1 Dual-Branch Baseline ($f_{\text{Dual}}$)

**Inputs:** $\mathbf{y} \in \mathbb{R}^M$ and $\mathbf{r} \in \mathbb{R}^4$

- **Branch 1 ($f_{\text{CS}}$):**  
  Linear($M \to 64$) → BatchNorm1d → ReLU → Dropout(0.2) → Linear($64 \to 16$) → BatchNorm1d → ReLU  
  ⟹ $\mathbf{z}_{cs} \in \mathbb{R}^{16}$

- **Branch 2 ($f_{\text{RR}}$):**  
  Linear($4 \to 16$) → BatchNorm1d → ReLU → Linear($16 \to 16$) → BatchNorm1d → ReLU  
  ⟹ $\mathbf{z}_{rr} \in \mathbb{R}^{16}$

- **Fusion:**  
  Concatenation $\mathbf{z}_{dual} = [\mathbf{z}_{cs} \parallel \mathbf{z}_{rr}] \in \mathbb{R}^{32}$

- **Classification Head:**  
  Linear($32 \to 16$) → BatchNorm1d → ReLU → Dropout(0.1) → Linear($16 \to 3$) → Softmax

---

### 5.2 Tri-Branch Proposed Architecture ($f_{\text{Tri}}$)

**Inputs:** $\mathbf{y} \in \mathbb{R}^M$, $\mathbf{r} \in \mathbb{R}^4$, and $\mathbf{h} \in \mathbb{R}^8$

- **Branches 1 & 2:** Identical to Dual-Branch configuration  
  ($\mathbf{z}_{cs} \in \mathbb{R}^{16}$, $\mathbf{z}_{rr} \in \mathbb{R}^{16}$)

- **Branch 3 ($f_{\text{HRV}}$):**  
  Linear($8 \to 16$) → BatchNorm1d → ReLU → Linear($16 \to 16$) → BatchNorm1d → ReLU  
  ⟹ $\mathbf{z}_{hrv} \in \mathbb{R}^{16}$

- **Balanced Latent Fusion:**  
  $\mathbf{z}_{fused} = [\mathbf{z}_{cs} \parallel \mathbf{z}_{rr} \parallel \mathbf{z}_{hrv}] \in \mathbb{R}^{48}$

- **Decision Head:**  
  Linear($48 \to 24$) → BatchNorm1d → ReLU → Dropout(0.1) → Linear($24 \to 3$) → Softmax

---

### 5.3 Class-Imbalance Regularization & Decision Calibration

To address extreme class imbalance across the **101,736 test beats** without collapsing sensitivity on minority classes:

- **Focal Loss Formulation:**  
  The model trains with focusing parameter $\gamma = 2.0$ and dampened class weighting factors $\alpha_c$:

  $$
  L_{\text{Focal}} = - \sum_{c=0}^{2} \alpha_c (1 - \hat{p}_c)^\gamma \log(\hat{p}_c)
  $$

  where $\alpha_c = \frac{N_{\text{total}}}{N_c}$, preventing majority class gradients from overwhelming the network.

- **Calibrated Decision Boundary:**  
  Standard argmax classification introduces false alarms for arrhythmia.  
  The prediction logic applies a tuned decision threshold $\tau = 0.65$ on the ARR probability output:

  $$
  \hat{y} =
  \begin{cases}
  1 \quad (\text{ARR}), & \text{if } \hat{p}_1 \geq 0.65 \\
  \arg\max_{c \in \{0,2\}} \hat{p}_c, & \text{otherwise}
  \end{cases}
  $$

---

## 6. Experimental Results & Performance Benchmarking

### 6.1 Diagnostic Resolution: Dual-Branch vs. Tri-Branch ($M=32$, 75% CR)

Evaluating on the complete independent test set (**101,736 beats**) reveals the impact of introducing the macro-scale autonomic branch ($f_{\text{HRV}}$):

| Metric                        | Dual-Branch Baseline ($y+r$) | Tri-Branch Architecture ($y+r+h$) | Relative Δ |
|-------------------------------|------------------------------|-----------------------------------|------------|
| Overall Test Accuracy         | 87.12%                       | 95.40%                            | +8.28%     |
| ARR Precision                 | 44.39%                       | 71.34%                            | +26.95%    |
| ARR F1-Score                  | 0.5842                       | 0.7765                            | +0.1923    |
| CHF Recall (Sensitivity)      | 78.28%                       | 96.88%                            | +18.60%    |
| CHF-to-NSR False Negatives    | 5,891 beats                  | 845 beats                         | −85.65%    |
| Macro Average F1-Score        | 0.7812                       | 0.9124                            | +0.1312    |

---

### 6.2 Scaling Across Sub-Nyquist Measurement Dimensions ($M$)

Evaluating performance across compression dimensions demonstrates the implicit regularization delivered by Gaussian random projections:

| Metric              | M=32 (CR=75%) | M=64 (CR=50%) | M=96 (CR=25%) | M=128 (Uncompressed) |
|---------------------|---------------|---------------|---------------|-----------------------|
| Overall Accuracy    | 95.40%        | 95.12%        | 94.88%        | 94.65%                |
| NSR Recall          | 95.54%        | 95.20%        | 94.90%        | 94.72%                |
| NSR Precision       | 97.53%        | 97.60%        | 97.45%        | 97.30%                |
| CHF Recall          | 96.88%        | 98.52%        | 98.60%        | 98.71%                |
| ARR F1-Score        | 0.7765        | 0.7690        | 0.7580        | 0.7510                |

**Key Observation (Implicit Regularization):**  
Peak overall diagnostic accuracy (95.40%) and ARR F1-score (0.7765) occur at the highest compression level ($M=32$).  
Sub-Nyquist random projection attenuates patient-specific, high-frequency baseline noise, curbing overfitting and improving generalizability compared to using raw, uncompressed 128-sample inputs.

---

## 7. Edge Hardware Complexity & Embedded Viability

To establish real-time feasibility on resource-constrained embedded microcontrollers (ARM Cortex-M4 / RISC-V at 64 MHz), parameter counts and Multiply-Accumulate operations (MACs) are strictly profiled.

### 7.1 Mathematical Complexity Breakdown ($M=32$)

| Functional Block              | Layer Specification | Trainable Parameters | Computational Workload (MACs/beat) |
|-------------------------------|---------------------|----------------------|------------------------------------|
| Morphology Branch ($f_{CS}$)  | 32→64→16            | 3,152                | 6,304                              |
| Rhythm Branch ($f_{RR}$)      | 4→16→16             | 352                  | 704                                |
| Autonomic Branch ($f_{HRV}$)  | 8→16→16             | 416                  | 832                                |
| Tri-Branch Decision Head       | 48→24→3             | 1,251                | 2,502                              |
| **Complete System**            | —                   | **5,171 (<6.5k)**    | **10,342 (≈10.5k)**                |

---

### 7.2 Architectural Advantages Over Existing Models

- **Ultra-Low Memory Footprint:**  
  At only **5,171 trainable parameters** (≈20.7 KB for 32-bit floats), the full model easily fits within standard on-chip SRAM cache (e.g., 96 KB SRAM on STM32 platforms).

- **Real-Time Latency:**  
  Processing 10.5k MACs on an embedded 64 MHz microcontroller requires **<0.3 ms per heartbeat**.  
  Since heartbeats occur approximately once every 800 ms, the core remains idle for >99.9% of each cardiac cycle, conserving battery life.

- **Payload Transmission Savings:**  
  By transmitting measurements at $M=32$ instead of raw Nyquist streams ($N=128$), the system achieves a **75% reduction in radio telemetry transmission energy**, addressing the primary source of power dissipation in wearable IoMT hardware.


## RAW CONTINUOUS ECG PROCESSING SCHEMATIC

### Input Signal
- **Raw Continuous ECG**

---

### Preprocessing Windows
- **Window (N=128 samples)**  
- **R-Peak Delta**  
- **60s Rolling Window**

---

### Multi-Scale Feature Extraction

**Micro-Scale: Compressed Morphology**
- Input: $\mathbf{y} \in \mathbb{R}^M$ (M=32)  
- Output: Compressed Vector Representation

**Meso-Scale: Local Rhythm**
- Input: $\mathbf{r} \in \mathbb{R}^4$  
- Output: Local Coupling Features

**Macro-Scale: Autonomic HRV**
- Input: $\mathbf{h} \in \mathbb{R}^8$  
- Output: Autonomic Heart Rate Variability Features



### Input Domains
- **Compressed Morphology (Micro-Scale):** $\mathbf{y} \in \mathbb{R}^M$ (M=32)  
- **Local Rhythm (Meso-Scale):** $\mathbf{r} \in \mathbb{R}^4$  
- **Autonomic HRV (Macro-Scale):** $\mathbf{h} \in \mathbb{R}^8$  

---

### Branch 1: Compressed Morphology ($f_{CS}$)
- Linear($M \to 64$)  
- BatchNorm1D(64)  
- ReLU, Dropout(0.2)  
- Linear($64 \to 16$)  
- BatchNorm1D(16)  
- ReLU  
⟹ $\mathbf{z}_{cs} \in \mathbb{R}^{16}$

---

### Branch 2: Local Rhythm ($f_{RR}$)
- Linear($4 \to 16$)  
- BatchNorm1D(16)  
- ReLU  
- Linear($16 \to 16$)  
- BatchNorm1D(16)  
- ReLU  
⟹ $\mathbf{z}_{rr} \in \mathbb{R}^{16}$

---

### Branch 3: Autonomic HRV ($f_{HRV}$)
- Linear($8 \to 16$)  
- BatchNorm1D(16)  
- ReLU  
- Linear($16 \to 16$)  
- BatchNorm1D(16)  
- ReLU  
⟹ $\mathbf{z}_{hrv} \in \mathbb{R}^{16}$

---

### Balanced Latent Fusion
Concatenation:  
$$\mathbf{z}_{fused} = [\mathbf{z}_{cs} \parallel \mathbf{z}_{rr} \parallel \mathbf{z}_{hrv}] \in \mathbb{R}^{48}$$

---

### Decision Head
- Linear($48 \to 24$)  
- BatchNorm1D(24)  
- ReLU, Dropout(0.1)  
- Linear($24 \to 3$)  
⟹ Output logits

---

### Final Prediction
- **Softmax Activation:** $\hat{\mathbf{p}} \in [0,1]^3$  
- **Calibrated Threshold ($\tau = 0.65$):**  
  - If $\hat{p}_1 \geq 0.65$ → Predict **ARR**  
  - Else → $\arg\max_{c \in \{0,2\}} \hat{p}_c$  
- **Predicted Class:** {NSR / ARR / CHF}
